<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_07_Generating_Synthetic_Faces_using_VAEs_and_Wasserstein_GANs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7: Generating Synthetic Faces using VAEs and Wasserstein GANs


**Instructor:** John Sipple

**Frameworks:** JAX, Optax, Equinox

##Overview

In this tutorial, we move from discriminative models to **Generative Models**. We will explore two highly influential architectures for image synthesis: the Variational Autoencoder (VAE) and the Generative Adversarial Network (GAN).

First, we will build a VAE, which forces an encoder to map inputs to a continuous probability distribution rather than a fixed point, allowing us to sample and interpolate new data.

Then, to overcome the VAE's tendency to produce blurry images, we will extract its learned Decoder and use it as the Generator in a **Wasserstein GAN (WGAN)**. By shifting from a probabilistic reconstruction loss to a competitive, game-theoretic approach, we will attempt to teach the network to generate much sharper, highly realistic facial textures.

##Key Objectives:

1. **The Reparameterization Trick**: Learn how to backpropagate through random sampling.

2. **VAE Loss Function**: Implement the Evidence Lower Bound (ELBO), combining Reconstruction Loss and Kullback-Leibler (KL) Divergence.

3. **Architecture Rendering**: Build the VAE in Equinox and dynamically render its spatial blueprint.

4. **Latent Space Analysis**: Visualize the distributions of the means ($\mu$) and variances ($\sigma^2$), and perform smooth interpolations between distinct faces.

5. **Transfer Learning**: Perform a deep copy of the VAE Decoder to act as the Generator in our WGAN.

6. **The Critic (Discriminator)**: Build a deep Convolutional Neural Network with Stride-1 convolutions, Max Pooling, and explicit JAX Dropout.

7. **Wasserstein Distance**: Replace the standard Min-Max probability objective with the Earth Mover's distance to prevent vanishing gradients and mode collapse.

8. **Adversarial Training**: Pre-train the Critic, then execute an imbalanced training loop to seek a Nash Equilibrium.

9. **Critic Evaluation**: Analyze the Critic's raw scores and squashed pseudo-probabilities on real vs. generated faces.

In [ ]:
# --- Setup and Installation ---
!pip install -q equinox optax jax jaxlib matplotlib numpy scikit-learn

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

# Set precision
jax.config.update("jax_enable_x64", False)
print(f"JAX Devices: {jax.devices()}")

# Part 1: The Olivetti Faces Dataset
We will load the dataset, normalize it, and reshape it into a standard format for Convolutional Neural Networks (CNNs): (`Batch`, `Channels`, `Height`, `Width`). Note that JAX/Equinox typically prefers channels-first natively, unlike standard TensorFlow.

In [ ]:
# --- 1. Load and Preprocess Data ---
print("Downloading Olivetti Faces dataset...")
dataset = fetch_olivetti_faces(shuffle=True, random_state=42)
faces = dataset.images # Shape: (400, 64, 64)

# Reshape to (Batch, Channels, Height, Width) for Equinox Conv2D
faces = faces.reshape(-1, 1, 64, 64)

# Train/Test Split
X_train, X_test = train_test_split(faces, test_size=0.1, random_state=42)
X_train = jnp.array(X_train)
X_test = jnp.array(X_test)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# --- 2. Visualize Sample Faces ---
def show_faces(images, n=5, title="Sample Real Faces"):
    fig, axes = plt.subplots(1, n, figsize=(n * 2, 2.5))
    for i in range(n):
        axes[i].imshow(images[i, 0], cmap='gray')
        axes[i].axis('off')
    plt.suptitle(title, fontsize=14)
    plt.show()

show_faces(X_train)

#Part 2: Defining the VAE Architecture

The VAE consists of three parts:

1. **Encoder**: Maps the input image $x$ to two latent vectors: $\mu$ and $\log(\sigma^2)$. We use $\log(\sigma^2)$ instead of $\sigma$ for numerical stability, as the network can output any real number without constraint.

2. **Reparameterization**: To sample $z$ while keeping the process differentiable, we use the trick:

$$z = \mu + \sigma \odot \epsilon$$

where $\epsilon \sim \mathbb{N}(0, I)$.Decoder: Maps the sampled $z$ back to the original image dimensions $\hat{x}$.

In [ ]:
# --- 1. Encoder ---
class Encoder(eqx.Module):
    layers: list
    fc_mu: eqx.nn.Linear
    fc_logvar: eqx.nn.Linear

    def __init__(self, latent_dim, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        self.layers = [
            eqx.nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, key=k1), # 32x32
            eqx.nn.Conv2d(32, 64, kernel_size=5, stride=2, padding=2, key=k2), # 16x16
            eqx.nn.Conv2d(64, 64, kernel_size=5, stride=2, padding=2, key=k3), # 8x8
        ]
        # Flatten size: 64 channels * 8 height * 8 width = 4096
        self.fc_mu = eqx.nn.Linear(4096, latent_dim, key=k4)

        k5, = jax.random.split(k4, 1)
        self.fc_logvar = eqx.nn.Linear(4096, latent_dim, key=k5)

    def __call__(self, x):
        for layer in self.layers:
            x = jax.nn.relu(layer(x))
        x = x.flatten()
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

# --- 2. Decoder ---
class Decoder(eqx.Module):
    fc_in: eqx.nn.Linear
    layers: list

    def __init__(self, latent_dim, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        self.fc_in = eqx.nn.Linear(latent_dim, 4096, key=k1)
        self.layers = [
            eqx.nn.ConvTranspose2d(64, 64, kernel_size=5, stride=2, padding=2, output_padding=1, key=k2), # 16x16
            eqx.nn.ConvTranspose2d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1, key=k3), # 32x32
            eqx.nn.ConvTranspose2d(32, 1, kernel_size=5, stride=2, padding=2, output_padding=1, key=k4)   # 64x64
        ]

    def __call__(self, z):
        x = jax.nn.relu(self.fc_in(z))
        x = x.reshape((64, 8, 8)) # Reshape back to feature maps
        for layer in self.layers[:-1]:
            x = jax.nn.relu(layer(x))
        # Final layer uses sigmoid to output pixel values in [0, 1]
        x = jax.nn.sigmoid(self.layers[-1](x))
        return x

# --- 3. Full VAE ---
class VAE(eqx.Module):
    encoder: Encoder
    decoder: Decoder

    def __init__(self, latent_dim, key):
        k1, k2 = jax.random.split(key)
        self.encoder = Encoder(latent_dim, key=k1)
        self.decoder = Decoder(latent_dim, key=k2)

    def reparameterize(self, mu, logvar, key):
        std = jnp.exp(0.5 * logvar)
        eps = jax.random.normal(key, shape=std.shape)
        return mu + eps * std

    def __call__(self, x, key):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar, key)
        recon_x = self.decoder(z)
        return recon_x, mu, logvar

# Instantiate Model
LATENT_DIM = 32
key = jax.random.PRNGKey(42)
vae_model = VAE(latent_dim=LATENT_DIM, key=key)

#Part 3: Rendering the VAE Architecture

Let's dynamically render the internal dimensions of the model to ensure our spatial reasoning maps correctly to the code.

In [ ]:
def render_vae_architecture(latent_dim):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axis('off')

    # Draw Helper
    def draw_block(x, y, w, h, text, color):
        ax.add_patch(patches.Rectangle((x, y), w, h, ec='black', fc=color, lw=2, zorder=2))
        ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=10, weight='bold')

    # Input
    draw_block(0, 3, 1, 2, "Input\n1x64x64", "#e6f2ff")
    ax.arrow(1, 4, 0.5, 0, head_width=0.2, fc='k', ec='k')

    # Encoder
    draw_block(1.5, 2.5, 2.5, 3, "Encoder\n(Conv2D Layers)\n\n16x32x32\n32x16x16\n64x8x8", "#add8e6")
    ax.arrow(4, 4, 0.5, 1, head_width=0.2, fc='k', ec='k')
    ax.arrow(4, 4, 0.5, -1, head_width=0.2, fc='k', ec='k')

    # Latent Space (Mu and Logvar)
    draw_block(4.5, 4.5, 1.5, 1, f"Mu ($\mu$)\ndim: {latent_dim}", "#ffb6c1")
    draw_block(4.5, 2.5, 1.5, 1, f"LogVar ($\sigma^2$)\ndim: {latent_dim}", "#ffb6c1")

    # Reparameterization
    ax.arrow(6, 5, 0.5, -0.8, head_width=0.2, fc='k', ec='k')
    ax.arrow(6, 3, 0.5, 0.8, head_width=0.2, fc='k', ec='k')
    ax.add_patch(patches.Circle((6.8, 4), 0.3, fc='yellow', ec='black', zorder=3))
    ax.text(6.8, 4, "$+$", ha='center', va='center', fontsize=14, weight='bold', zorder=4)
    ax.text(6.8, 4.5, "Sample $z$", ha='center', va='center', fontsize=10, weight='bold')

    # Epsilon Injection
    ax.arrow(6.8, 2.5, 0, 1.2, head_width=0.1, fc='red', ec='red', linestyle='--')
    ax.text(6.8, 2.2, "$\epsilon \sim \mathcal{N}(0, I)$", ha='center', va='center', color='red')

    # Decoder
    ax.arrow(7.1, 4, 0.4, 0, head_width=0.2, fc='k', ec='k')
    draw_block(7.5, 2.5, 2.5, 3, "Decoder\n(ConvTranspose2D)\n\n32x16x16\n16x32x32\n1x64x64", "#98fb98")

    # Output
    ax.arrow(10, 4, 0.5, 0, head_width=0.2, fc='k', ec='k')
    draw_block(10.5, 3, 1, 2, "Output\n1x64x64", "#e6f2ff")

    plt.title("Variational Autoencoder (VAE) Spatial Blueprint", fontsize=16)
    plt.xlim(-0.5, 12)
    plt.ylim(1, 7)
    plt.show()

render_vae_architecture(LATENT_DIM)

#Part 4: The Loss Function & Training Loop

The VAE optimizes the Evidence Lower Bound (ELBO). We want to minimize the negative ELBO, which splits into two terms:

1. **Reconstruction Loss**: Measures how well the output matches the input (using Mean Squared Error or Binary Cross-Entropy). We'll use MSE here.

2. **KL Divergence:** Acts as a regularizer, forcing the distributions learned by the encoder to approximate a standard normal distribution $\mathbb{N}(0, I)$.

$$KL = -0.5 \sum (1 + \log(\sigma^2) - \mu^2 - \sigma^2)$$

In [ ]:
BATCH_SIZE = 32
EPOCHS = 1000
LEARNING_RATE = 1e-4

optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(eqx.filter(vae_model, eqx.is_array))

# --- 1. Loss Function ---
@eqx.filter_value_and_grad
def compute_loss(model, x_batch, key_batch):
    # Vectorize the model over the batch dimension
    batched_vae = jax.vmap(model, in_axes=(0, 0))
    recon_x, mu, logvar = batched_vae(x_batch, key_batch)

    # Reconstruction Loss (MSE)
    recon_loss = jnp.mean(jnp.sum(jnp.square(recon_x - x_batch), axis=(1,2,3)))

    # KL Divergence
    kl_loss = -0.5 * jnp.mean(jnp.sum(1 + logvar - jnp.square(mu) - jnp.exp(logvar), axis=1))

    # Total loss (we often weight the KL term, but standard is 1.0)
    return recon_loss + kl_loss

# --- 2. Train Step ---
@eqx.filter_jit
def train_step(model, opt_state, x_batch, key_batch):
    loss, grads = compute_loss(model, x_batch, key_batch)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

@eqx.filter_jit
def eval_step(model, x_batch, key_batch):
    # Just forward pass for test loss
    batched_vae = jax.vmap(model, in_axes=(0, 0))
    recon_x, mu, logvar = batched_vae(x_batch, key_batch)
    recon_loss = jnp.mean(jnp.sum(jnp.square(recon_x - x_batch), axis=(1,2,3)))
    kl_loss = -0.5 * jnp.mean(jnp.sum(1 + logvar - jnp.square(mu) - jnp.exp(logvar), axis=1))
    return recon_loss + kl_loss

# --- 3. Training Loop ---
print("Training VAE...")
train_losses = []
test_losses = []
rng = jax.random.PRNGKey(101)

for epoch in range(EPOCHS):
    rng, step_key = jax.random.split(rng)

    # Shuffle
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]

    epoch_loss = 0.0
    n_batches = max(1, len(X_train) // BATCH_SIZE)

    for i in range(n_batches):
        bx = X_train_shuffled[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        b_keys = jax.random.split(step_key, bx.shape[0])

        vae_model, opt_state, loss = train_step(vae_model, opt_state, bx, b_keys)
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / n_batches

    # Eval
    test_keys = jax.random.split(step_key, X_test.shape[0])
    test_loss = eval_step(vae_model, X_test, test_keys).item()

    train_losses.append(avg_train_loss)
    test_losses.append(test_loss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.2f} | Test Loss: {test_loss:.2f}")

# --- 4. Plot Losses ---
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(test_losses, label='Test Loss', color='orange')
plt.title("VAE Training & Test Loss (ELBO)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#Part 5: Investigating the Latent Space (Means and Variances)

Because of the KL Divergence term, our VAE is forced to pack its latent variables closely around zero with a variance near 1. Let's pass our test set through the encoder and visualize the resulting distributions to confirm the regularizer worked.

In [ ]:
# Pass the test set through the encoder
batched_encoder = jax.vmap(vae_model.encoder)
mu_test, logvar_test = batched_encoder(X_test)

mu_test = np.array(mu_test)
logvar_test = np.array(logvar_test)
var_test = np.exp(logvar_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot Mean distribution
axes[0].hist(mu_test.flatten(), bins=50, color='purple', alpha=0.7)
axes[0].set_title(f"Distribution of Latent Means ($\mu$)")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Frequency")
axes[0].axvline(0, color='black', linestyle='--')

# Plot Variance distribution
axes[1].hist(var_test.flatten(), bins=50, color='green', alpha=0.7)
axes[1].set_title("Distribution of Latent Variances ($\sigma^2$)")
axes[1].set_xlabel("Value")
axes[1].axvline(1, color='black', linestyle='--')

plt.tight_layout()
plt.show()

#Part 6: Generative Synthesis (Sampling New Faces)

Since we successfully structured the latent space to approximate $\mathbb{N}(0, I)$, we can now entirely bypass the encoder. By randomly sampling vectors from a standard normal distribution and feeding them into the Decoder, we generate faces of people that do not exist!

In [ ]:
def generate_synthetic_faces(model, num_faces=10, key_seed=999):
    rng = jax.random.PRNGKey(key_seed)

    # 1. Sample pure noise from a Standard Normal Distribution
    z_sampled = jax.random.normal(rng, (num_faces, LATENT_DIM))

    # 2. Pass the noise directly into the Decoder
    batched_decoder = jax.vmap(model.decoder)
    synthetic_images = batched_decoder(z_sampled)

    # 3. Render
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()
    for i in range(num_faces):
        axes[i].imshow(synthetic_images[i, 0], cmap='gray')
        axes[i].set_title(f"Generated Face {i+1}", fontsize=10)
        axes[i].axis('off')

    plt.suptitle("Synthetically Generated Olivetti Faces via VAE Decoder", fontsize=16)
    plt.tight_layout()
    plt.show()

# Generate and render 10 new faces
generate_synthetic_faces(vae_model, num_faces=10, key_seed=1293)

#Part 7: Latent Space Interpolation (Morphing Faces)

One of the defining features of a Variational Autoencoder is that its latent space is continuous. Because the KL Divergence term forces the encodings to overlap and pack tightly around the origin, there are no "empty holes" in the manifold.

This means if we take the latent vector for Person A ($\mu_A$) and the latent vector for Person B ($\mu_B$), we can mathematically walk a straight line between them. If we decode the points along that line, we should see a smooth, plausible transition (a morph) from one identity to the other.Mathematically, we use linear interpolation (Lerp):

$$z_{interp} = (1 - \alpha) \cdot \mu_A + \alpha \cdot \mu_B$$

Where $\alpha$ smoothly transitions from $0.0$ to $1.0$.

In [ ]:
def interpolate_faces(model, img_a, img_b, steps=8):
    # 1. Encode both images to get their latent means
    # We use vmap to process both images in parallel
    batched_encoder = jax.vmap(model.encoder)
    mu, _ = batched_encoder(jnp.stack([img_a, img_b]))
    mu_a, mu_b = mu[0], mu[1]

    # 2. Create the interpolation steps
    # alphas shape: (steps, 1) to allow broadcasting against the latent vector
    alphas = jnp.linspace(0, 1, steps)[:, None]

    # 3. Calculate the intermediate latent vectors
    z_interp = (1 - alphas) * mu_a + alphas * mu_b

    # 4. Decode the intermediate vectors back into images
    batched_decoder = jax.vmap(model.decoder)
    synthetic_morphs = batched_decoder(z_interp)

    # 5. Render the results
    fig, axes = plt.subplots(1, steps + 2, figsize=(15, 3))

    # Show Original Image A
    axes[0].imshow(img_a[0], cmap='gray')
    axes[0].set_title("Original A", weight='bold')
    axes[0].axis('off')

    # Show Interpolations
    for i in range(steps):
        axes[i+1].imshow(synthetic_morphs[i, 0], cmap='gray')
        axes[i+1].set_title(f"$\\alpha$ = {alphas[i][0]:.2f}")
        axes[i+1].axis('off')

    # Show Original Image B
    axes[-1].imshow(img_b[0], cmap='gray')
    axes[-1].set_title("Original B", weight='bold')
    axes[-1].axis('off')

    plt.suptitle("Walking the Continuous Latent Manifold", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

# Select two distinct faces from the test set
# (You can change these indices to see different morphs!)
face_A = X_test[0]
face_B = X_test[5]

print("Generating latent interpolation...")
interpolate_faces(vae_model, face_A, face_B, steps=8)

# Part 8: Generative Adversarial Networks

While VAEs learn by optimizing a lower bound on the data likelihood (ELBO), Generative Adversarial Networks (GANs) take a game-theoretic approach. Instead of calculating a direct reconstruction loss, we train two networks to play a continuous game of cat-and-mouse.

## The Generator and the Discriminator

* **The Generator**($G$): This takes the place of our VAE's Decoder. It receives a random noise vector $z$ and tries to synthesize a realistic image.

* T**he Discriminator** ($D$): This is a new Convolutional Neural Network acting as an art critic or detective. It looks at images and tries to classify them as either Real (from the training set) or Fake (created by the Generator).

Let's reuse our VAE's Decoder as the Generator, and build a new CNN for the Discriminator.

In [ ]:
# @title Create Generator and Critic
import copy
key_g, key_d = jax.random.split(jax.random.PRNGKey(492))
reuse_existing_decoder = True # @param {"type":"boolean"}
# --- 1. The Generator (Reusing the VAE Decoder) ---

if reuse_existing_decoder:
  # We use deepcopy to explicitly create an independent instance of the decoder.
  # This gives the GAN a massive head start on understanding facial geometry
  # while conceptually detaching it from the original VAE model.
  generator = copy.deepcopy(vae_model.decoder)
else:
  # We instantiate a fresh Decoder to act as our GAN Generator
  key_g, key_d = jax.random.split(jax.random.PRNGKey(42))
  generator = Decoder(latent_dim=LATENT_DIM, key=key_g)

# --- 2. The Discriminator / Critic ---
class Critic(eqx.Module):
    convs: list
    pools: list
    linear1: eqx.nn.Linear
    linear2: eqx.nn.Linear
    linear3: eqx.nn.Linear

    def __init__(self, key):
        keys = jax.random.split(key, 7)

        # 3 Convolutional Layers (Kernel 5, Stride 1, Padding 2 to keep dimensions even)
        self.convs = [
            eqx.nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2, key=keys[0]),
            eqx.nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2, key=keys[1]),
            eqx.nn.Conv2d(64, 64, kernel_size=5, stride=1, padding=2, key=keys[2])
        ]

        # Max pooling to reduce spatial dimensions since stride is now 1
        self.pools = [eqx.nn.MaxPool2d(kernel_size=2, stride=2) for _ in range(3)]

        # After three 2x2 pools, the 64x64 image shrinks to 8x8.
        # Flattened size: 64 channels * 8 * 8 = 4096
        self.linear1 = eqx.nn.Linear(4096, 512, key=keys[3])
        self.linear2 = eqx.nn.Linear(512, 128, key=keys[4])
        self.linear3 = eqx.nn.Linear(128, 1, key=keys[5])



    def __call__(self, x):
        """
        inference: If True, dropout is bypassed (used during final evaluation)
        key: Required if inference=False to generate the random dropout mask.
        """
        for conv, pool in zip(self.convs, self.pools):
            x = jax.nn.leaky_relu(conv(x), negative_slope=0.2)
            x = pool(x)

        x = x.flatten()

        # Linear Layers with Dropout
        x = jax.nn.leaky_relu(self.linear1(x), negative_slope=0.2)
        x = jax.nn.leaky_relu(self.linear2(x), negative_slope=0.2)


        return self.linear3(x)

# Instantiate the new Critic
key_d = jax.random.PRNGKey(42)
discriminator = Critic(key=key_d)

#Part 9: Objective Functions and the Wasserstein Distance

##The Standard Min-Max Objective

In a standard GAN, the Discriminator outputs a probability (using a sigmoid) of whether an image is real. The two networks play a zero-sum game optimized by the Min-Max objective:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_{z}}[\log(1 - D(G(z)))]$$

* **Discriminator goal:** Maximize the equation (push $D(x)$ to 1, and $D(G(z))$ to 0).

* **Generator goal:** Minimize the equation (push $D(G(z))$ to 1 to fool $D$).

**The Problem:** Standard GANs suffer from vanishing gradients and "mode collapse" (where the generator produces the exact same image over and over). This happens because the implicit Jensen-Shannon Divergence used by standard GANs provides terrible gradients when the real and fake distributions don't overlap.

##The Wasserstein Distance (WGAN)
To fix this, we replace the KL/JS divergence with the Wasserstein Distance (also known as the Earth Mover's distance). It measures the minimum "cost" to physically move the mass of the generated distribution to match the real distribution.

In a WGAN:

1. The Discriminator is renamed to a **Critic**. It outputs a raw score (real number), not a probability.

2. **Critic Loss**: Maximize the difference between scores of real and fake images.

$$L_D = \mathbb{E}[C(G(z))] - \mathbb{E}[C(x)]$$

3. **Generator Loss**: Maximize the critic's score for fake images.

$$L_G = -\mathbb{E}[C(G(z))]$$

4. **Lipschitz Constraint**: The mathematics of WGAN require the Critic to be relatively smooth. We enforce this using **Weight Clipping** (clipping the Critic's weights to a small range like $[-0.01, 0.01]$ after every step).

In [ ]:
# WGANs typically use RMSProp instead of Adam to avoid momentum instability with weight clipping
LR_G_WGAN = 5e-7
LR_D_WGAN = 5e-5
opt_G = optax.rmsprop(LR_G_WGAN)
opt_D = optax.rmsprop(LR_D_WGAN)

opt_state_G = opt_G.init(eqx.filter(generator, eqx.is_array))
opt_state_D = opt_D.init(eqx.filter(discriminator, eqx.is_array))

# --- 1. WGAN Loss Functions ---
@eqx.filter_value_and_grad
def compute_loss_D(critic, generator, real_x, z):
    fake_x = jax.vmap(generator)(z)

    score_real = jax.vmap(critic)(real_x)
    score_fake = jax.vmap(critic)(fake_x)

    # Critic wants to maximize (score_real - score_fake), so we minimize the negative
    loss_D = jnp.mean(score_fake) - jnp.mean(score_real)
    return loss_D

@eqx.filter_value_and_grad
def compute_loss_G(generator, critic, z):
    fake_x = jax.vmap(generator)(z)
    score_fake = jax.vmap(critic)(fake_x)

    # Generator wants to maximize Critic's score of fake images
    loss_G = -jnp.mean(score_fake)
    return loss_G

# --- 2. Weight Clipping Utility ---
def clip_weights(model, clip_value=0.01):
    def clip_array(x):
        if isinstance(x, jax.Array):
            return jnp.clip(x, -clip_value, clip_value)
        return x
    return jax.tree_util.tree_map(clip_array, model)

# --- 3. Training Steps ---
@eqx.filter_jit
def step_D(critic, generator, opt_state_d, real_x, z):
    loss_d, grads_d = compute_loss_D(critic, generator, real_x, z)
    updates_d, opt_state_d = opt_D.update(grads_d, opt_state_d, critic)
    critic = eqx.apply_updates(critic, updates_d)
    # Apply weight clipping to satisfy 1-Lipschitz continuity
    critic = clip_weights(critic, 0.01)
    return critic, opt_state_d, loss_d

@eqx.filter_jit
def step_G(generator, critic, opt_state_g, z):
    loss_g, grads_g = compute_loss_G(generator, critic, z)
    updates_g, opt_state_g = opt_G.update(grads_g, opt_state_g, generator)
    generator = eqx.apply_updates(generator, updates_g)
    return generator, opt_state_g, loss_g

Part 9.5: Pre-training the Critic
Before we start the simultaneous game of cat-and-mouse, we are going to give the Critic a head start.

Because we are using a Wasserstein GAN (WGAN), the mathematics require the Critic to approximate the true Earth Mover's distance. If the Critic is poorly trained, the gradients it passes to the Generator will be meaningless garbage.

To fix this, we will freeze the Generator (which currently outputs the blurry faces it learned during the VAE phase) and train only the Critic for a few epochs. It will learn exactly how to differentiate a real Olivetti face from a blurry VAE face. Once the Critic is sharp, we will unleash the Generator to try and fool it.

In [ ]:
PRETRAIN_EPOCHS = 1000

print("Pre-training the Critic (Generator is frozen)...")
pretrain_history_D = []
rng = jax.random.PRNGKey(404)

for epoch in range(PRETRAIN_EPOCHS):
    rng, step_key = jax.random.split(rng)

    # Shuffle the real data
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]

    n_batches = max(1, len(X_train) // BATCH_SIZE)
    epoch_loss_d = 0.0

    for i in range(n_batches):
        bx = X_train_shuffled[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        current_batch_size = bx.shape[0]

        # 1. Sample pure noise
        rng, z_key = jax.random.split(rng)
        z = jax.random.normal(z_key, (current_batch_size, LATENT_DIM))

        # 2. Update ONLY the Critic (step_D handles the forward pass of G internally)
        discriminator, opt_state_D, loss_d = step_D(discriminator, generator, opt_state_D, bx, z)
        epoch_loss_d += loss_d.item()

    avg_pretrain_loss = epoch_loss_d / n_batches
    pretrain_history_D.append(avg_pretrain_loss)

    print(f"Pre-train Epoch {epoch+1}/{PRETRAIN_EPOCHS} | Critic Loss: {avg_pretrain_loss:.6f}")

# Plot the Critic's pre-training loss
plt.figure(figsize=(8, 4))
plt.plot(pretrain_history_D, label='Critic Pre-training Loss', color='red', marker='o')
plt.title("Critic Pre-training (Learning to spot VAE blurriness)")
plt.xlabel("Epoch")
plt.ylabel("Wasserstein Loss Proxy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import jax.nn

def evaluate_critic_probabilities(gen_model, critic_model, real_images, num_images=10, key_seed=111):
    rng = jax.random.PRNGKey(key_seed)

    # 1. Select Real Images and score them
    real_sample = X_test[:num_images]
    batched_critic = jax.vmap(critic_model)

    # Get raw scores and squash them to probabilities [0, 1]
    raw_real_scores = batched_critic(real_sample)
    real_probs = jax.nn.sigmoid(raw_real_scores)

    # 2. Generate Fake Images and score them
    z = jax.random.normal(rng, (num_images, LATENT_DIM))
    batched_gen = jax.vmap(gen_model)
    fake_sample = batched_gen(z)

    # Get raw scores and squash them to probabilities [0, 1]
    raw_fake_scores = batched_critic(fake_sample)
    fake_probs = jax.nn.sigmoid(raw_fake_scores)

    # 3. Render the results
    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 2, 5.5))

    # Plot Real Images (Top Row)
    for i in range(num_images):
        axes[0, i].imshow(real_sample[i, 0], cmap='gray')
        # Display as a percentage
        prob_pct = real_probs[i].item() * 100
        axes[0, i].set_title(f"Real\nProb: {prob_pct:.1f}%", color='darkgreen', fontsize=11, weight='bold')
        axes[0, i].axis('off')

    # Plot Fake Images (Bottom Row)
    for i in range(num_images):
        axes[1, i].imshow(fake_sample[i, 0], cmap='gray')
        # Display as a percentage
        prob_pct = fake_probs[i].item() * 100
        axes[1, i].set_title(f"Fake\nProb: {prob_pct:.1f}%", color='darkred', fontsize=11, weight='bold')
        axes[1, i].axis('off')

    plt.suptitle("Critic's Judgment: 'Pseudo-Probabilities' via Sigmoid Squashing", fontsize=16, y=1.05)

    # Add row labels
    fig.text(0.1, 0.75, 'Real Data', va='center', ha='right', fontsize=14, weight='bold', rotation=90)
    fig.text(0.1, 0.25, 'Generator', va='center', ha='right', fontsize=14, weight='bold', rotation=90)

    plt.tight_layout(rect=[0.12, 0, 1, 1])
    plt.show()

# Execute the evaluation using our test set
print("Extracting Critic probabilities...")
evaluate_critic_probabilities(generator, discriminator, X_test, num_images=10, key_seed=42)

#Part 10: Adversarial Training Loop


##Executing Training

Training a GAN requires a delicate balance. In WGANs, the Critic must be trained to optimality to provide a reliable Wasserstein gradient to the Generator. Therefore, we will train the Critic **5 times** for every 1 time we train the Generator.

(Note: Because GANs are notoriously hard to train and require thousands of epochs for high-fidelity images, we will run a shorter loop here to demonstrate the learning dynamics.)

In [ ]:
EPOCHS_GAN = 1200
N_CRITIC = 1 # Train critic 1 times per generator step

print("Training WGAN...")
history_D = []
history_G = []
rng = jax.random.PRNGKey(777)

for epoch in range(EPOCHS_GAN):
    rng, step_key = jax.random.split(rng)

    # Shuffle
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]

    n_batches = max(1, len(X_train) // BATCH_SIZE)
    epoch_loss_d = 0.0
    epoch_loss_g = 0.0
    g_steps = 0

    for i in range(n_batches):
        bx = X_train_shuffled[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        current_batch_size = bx.shape[0]

        # 1. Train Critic
        rng, z_key = jax.random.split(rng)
        z = jax.random.normal(z_key, (current_batch_size, LATENT_DIM))
        discriminator, opt_state_D, loss_d = step_D(discriminator, generator, opt_state_D, bx, z)
        epoch_loss_d += loss_d.item()

        # 2. Train Generator (only every N_CRITIC steps)
        if i % N_CRITIC == 0:
            rng, z_key = jax.random.split(rng)
            z = jax.random.normal(z_key, (current_batch_size, LATENT_DIM))
            generator, opt_state_G, loss_g = step_G(generator, discriminator, opt_state_G, z)
            epoch_loss_g += loss_g.item()
            g_steps += 1

    history_D.append(epoch_loss_d / n_batches)
    history_G.append(epoch_loss_g / max(1, g_steps))

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS_GAN} | Critic Loss: {history_D[-1]:.6f} | Gen Loss: {history_G[-1]:.6f}")

# Plotting the losses
plt.figure(figsize=(10, 4))
plt.plot(history_D, label='Critic Loss (Wasserstein Distance proxy)', color='red', alpha=0.8)
plt.plot(history_G, label='Generator Loss', color='blue', alpha=0.8)
plt.title("WGAN Adversarial Training Dynamics")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Interpreting the Loss and Nash Equilibrium
Unlike the VAE where the loss goes down smoothly towards zero, GAN losses look chaotic.

Because the networks are playing a zero-sum game, they are searching for a **Nash Equilibrium**. This is a state where neither the Generator nor the Discriminator can improve their standing without the other network changing its strategy.

In a perfectly trained standard GAN, the Discriminator is completely fooled and guesses exactly 0.5 (50%) for every image. In a WGAN, the Critic's loss measures the estimated Wasserstein distance between the real and fake distributions; as training progresses and the Generator improves, this distance should gradually shrink toward zero.

##Generating Synthetic Images

Now, let's use our battle-hardened Generator to synthesize completely new faces by sampling from the latent space $\mathscr{N}(0, I)$, exactly as we did with the VAE.

In [ ]:
def generate_gan_faces(gen_model, num_faces=10, key_seed=555):
    rng = jax.random.PRNGKey(key_seed)

    # Sample pure noise
    z_sampled = jax.random.normal(rng, (num_faces, LATENT_DIM))

    # Pass through the Generator
    batched_gen = jax.vmap(gen_model)
    synthetic_images = batched_gen(z_sampled)

    # Render
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()
    for i in range(num_faces):
        axes[i].imshow(synthetic_images[i, 0], cmap='gray')
        axes[i].set_title(f"GAN Face {i+1}", fontsize=10)
        axes[i].axis('off')

    plt.suptitle("Synthetically Generated Olivetti Faces via WGAN Generator", fontsize=16)
    plt.tight_layout()
    plt.show()

# Generate and render
generate_gan_faces(generator, num_faces=10, key_seed=8888)